# Autoencoder → Representation → Clustering → Evaluation

**Objective:**
1. Build a CNN autoencoder (encoder → 128D latent → decoder)
2. Train it to reconstruct WikiArt images
3. Extract latent embeddings
4. Reduce dimensionality (PCA → 2D for visualization)
5. Perform clustering (KMeans)
6. Compare clusters with `style` and `artist` labels (ARI, NMI, purity)

## 0. Imports and Setup

In [ ]:
import os
from pathlib import Path
import time
from contextlib import nullcontext

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T

from datasets import load_dataset, Image as HFImage
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from tqdm import tqdm

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

## 1. Load Local Parquet Dataset

In [ ]:
# Data dir: put parquet files at ./dataset1/data or set env DATA_DIR
DATA_DIR = Path(os.getenv("DATA_DIR", "dataset1/data")).resolve()
print("Data dir:", DATA_DIR)

files = sorted(str(p) for p in DATA_DIR.rglob("*.parquet"))
print("Found parquet files:", len(files))
if not files:
    raise SystemExit("No .parquet files found under ./dataset1/data. Upload/mount your data there.")

# Load dataset from local parquet
ds_full = load_dataset("parquet", data_files=files, split="train").cast_column("image", HFImage())
print("Total records:", len(ds_full))
print("Columns:", ds_full.column_names)

# Persist split indices so future runs reuse the same split
ARTIFACTS_DIR = Path("artifacts"); ARTIFACTS_DIR.mkdir(exist_ok=True)
SPLITS_PATH = ARTIFACTS_DIR / "dataset_splits.json"

import json

if SPLITS_PATH.exists():
    print("Loading existing split indices...")
    split_indices = json.loads(SPLITS_PATH.read_text())
else:
    print("Creating new 80/10/10 split (stratified by 'style' if available)...")
    ds_full = ds_full.add_column("idx", list(range(len(ds_full))))
    try:
        s1 = ds_full.train_test_split(test_size=0.20, stratify_by_column="style", seed=42)
        s2 = s1["test"].train_test_split(test_size=0.50, stratify_by_column="style", seed=42)
    except Exception as e:
        print("Stratified split failed, falling back to random split:", e)
        s1 = ds_full.train_test_split(test_size=0.20, seed=42)
        s2 = s1["test"].train_test_split(test_size=0.50, seed=42)

    train_ds, val_ds, test_ds = s1["train"], s2["train"], s2["test"]
    split_indices = {
        "train": train_ds["idx"] if "idx" in train_ds.column_names else train_ds.add_column("idx", list(range(len(train_ds))))["idx"],
        "val":   val_ds["idx"] if "idx" in val_ds.column_names else val_ds.add_column("idx", list(range(len(val_ds))))["idx"],
        "test":  test_ds["idx"] if "idx" in test_ds.column_names else test_ds.add_column("idx", list(range(len(test_ds))))["idx"],
    }
    SPLITS_PATH.write_text(json.dumps(split_indices))
    print("Saved split indices to", SPLITS_PATH)

# Rebuild splits from indices
train_ds = ds_full.select(split_indices["train"])
val_ds   = ds_full.select(split_indices["val"])
test_ds  = ds_full.select(split_indices["test"])
print(f"Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}")

## 2. Transforms and Dataset Wrapper

In [ ]:
# Speed knobs
IMG_SIZE = int(os.getenv("IMG_SIZE", "128"))  # 128 is faster on Colab CPU/GPU
LATENT_DIM = int(os.getenv("LATENT_DIM", "128"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "64" if torch.cuda.is_available() else "32"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "2"))  # Colab Linux: 2 workers is OK

# Subset for quick iterations (0 = use all)
MAX_TRAIN = int(os.getenv("MAX_TRAIN", "0"))
MAX_VAL   = int(os.getenv("MAX_VAL", "0"))
MAX_TEST  = int(os.getenv("MAX_TEST", "0"))

def maybe_subset(ds, limit):
    return ds.select(range(min(limit, len(ds)))) if (limit and limit > 0) else ds

train_ds_sub = maybe_subset(train_ds, MAX_TRAIN)
val_ds_sub   = maybe_subset(val_ds,   MAX_VAL)
test_ds_sub  = maybe_subset(test_ds,  MAX_TEST)

print(f"Using train={len(train_ds_sub)} val={len(val_ds_sub)} test={len(test_ds_sub)}")

# Transforms
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # [-1, 1]
])

class WikiArtDataset(Dataset):
    """Wrap HF dataset, apply transforms, return image + labels."""
    def __init__(self, hf_dataset, transform=None):
        self.hf_dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        img = item["image"]  # PIL Image
        if self.transform:
            img = self.transform(img)
        style = item.get("style", "unknown")
        artist = item.get("artist", "unknown")
        return img, style, artist

pin_mem = torch.cuda.is_available()
train_loader = DataLoader(WikiArtDataset(train_ds_sub, transform=transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin_mem, persistent_workers=NUM_WORKERS>0)
val_loader   = DataLoader(WikiArtDataset(val_ds_sub, transform=transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin_mem, persistent_workers=NUM_WORKERS>0)
test_loader  = DataLoader(WikiArtDataset(test_ds_sub, transform=transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin_mem, persistent_workers=NUM_WORKERS>0)

print(f"Batches -> train:{len(train_loader)} val:{len(val_loader)} test:{len(test_loader)}")

## 3. Autoencoder Definition (CNN)

In [ ]:
class Autoencoder(nn.Module):
    """CNN Autoencoder: 3xIMG_SIZExIMG_SIZE -> latent -> 3xIMG_SIZExIMG_SIZE."""
    def __init__(self, latent_dim=128, img_size=128):
        super().__init__()
        s = img_size // 16  # after 4 downsamples (stride=2)
        self._s = s

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),   # -> 32 x (img/2)
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),  # -> 64 x (img/4)
            nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(), # -> 128 x (img/8)
            nn.Conv2d(128, 256, 4, 2, 1), nn.ReLU(),# -> 256 x (img/16)
            nn.Flatten(),
            nn.Linear(256 * s * s, latent_dim)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256 * s * s),
            nn.Unflatten(1, (256, s, s)),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

model = Autoencoder(latent_dim=LATENT_DIM, img_size=IMG_SIZE).to(device)
print(model)

## 4. Autoencoder Training

In [ ]:
EPOCHS = int(os.getenv("EPOCHS", "6"))
LR = float(os.getenv("LR", "1e-3"))

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
amp_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0

    for imgs, _, _ in tqdm(train_loader, desc=f"Train {epoch}/{EPOCHS}", ncols=100):
        imgs = imgs.to(device, non_blocking=True)

        with amp_ctx:
            recon, _ = model(imgs)
            loss = criterion(recon, imgs)

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= max(1, len(train_loader.dataset))

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, _, _ in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            recon, _ = model(imgs)
            loss = criterion(recon, imgs)
            val_loss += loss.item() * imgs.size(0)
    val_loss /= max(1, len(val_loader.dataset))

    dt = time.time() - t0
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | time={dt/60:.2f} min")

print("Training finished")

In [ ]:
## 5. Save Model and Reconstruction Check

## 6. Extract Embeddings

In [ ]:
# Extract latent embeddings over the training set
model.eval()
embeddings, all_styles, all_artists = [], [], []
with torch.no_grad():
    for imgs, styles, artists in tqdm(train_loader, desc="Embedding", ncols=100):
        imgs = imgs.to(device, non_blocking=True)
        _, z = model(imgs)
        embeddings.append(z.cpu().numpy())
        all_styles.extend(styles)
        all_artists.extend(artists)

embeddings = np.concatenate(embeddings, axis=0)
print("Embeddings shape:", embeddings.shape)

## 7. PCA Dimensionality Reduction

In [ ]:
# PCA 50D for clustering + 2D for visualization
pca50 = PCA(n_components=50, random_state=42)
emb_50d = pca50.fit_transform(embeddings)

pca2 = PCA(n_components=2, random_state=42)
emb_2d = pca2.fit_transform(embeddings)
print(f"PCA -> 50D: {emb_50d.shape} | 2D: {emb_2d.shape}")

## 8. KMeans Clustering

In [ ]:
# Determine number of clusters from unique styles (bounded 5-30)
unique_styles = set(all_styles)
k = max(5, min(30, len(unique_styles)))
print(f"Unique styles: {len(unique_styles)} -> K={k}")

kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(emb_50d)

print(f"Cluster counts: {Counter(cluster_labels)}")

## 9. Evaluation Metrics (ARI, NMI, Purity)

In [ ]:
# Define purity metric
def purity_score(pred_clusters, true_labels):
    """Purity = sum over clusters of the dominant class count / total."""
    from collections import defaultdict
    groups = defaultdict(list)
    for c, t in zip(pred_clusters, true_labels):
        groups[int(c)].append(t)
    total = len(true_labels)
    correct = sum(Counter(v).most_common(1)[0][1] for v in groups.values())
    return correct / max(total, 1)

# Convert to arrays
style_arr = np.array(all_styles)
artist_arr = np.array(all_artists)

# Compute metrics vs style
ari_style = adjusted_rand_score(style_arr, cluster_labels)
nmi_style = normalized_mutual_info_score(style_arr, cluster_labels)
pur_style = purity_score(cluster_labels, all_styles)

# Compute metrics vs artist
ari_artist = adjusted_rand_score(artist_arr, cluster_labels)
nmi_artist = normalized_mutual_info_score(artist_arr, cluster_labels)
pur_artist = purity_score(cluster_labels, all_artists)

print("\n=== STYLE ===")
print(f"ARI: {ari_style:.4f}")
print(f"NMI: {nmi_style:.4f}")
print(f"Purity: {pur_style:.4f}")

print("\n=== ARTIST ===")
print(f"ARI: {ari_artist:.4f}")
print(f"NMI: {nmi_artist:.4f}")
print(f"Purity: {pur_artist:.4f}")

## 10. Visualization and Save Artifacts

In [ ]:
# Scatter plot by cluster
plt.figure(figsize=(10, 7))
scatter = plt.scatter(emb_2d[:,0], emb_2d[:,1], c=cluster_labels, cmap='tab20', s=10, alpha=0.7)
plt.colorbar(scatter, label="Cluster ID")
plt.title(f"KMeans Clustering (K={k}) on PCA 2D")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout()
plot_path = ARTIFACTS_DIR / "clusters_pca2d.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print("Plot saved:", plot_path.resolve())

# Save arrays and CSV
np.savez(ARTIFACTS_DIR / "embeddings_autoencoder.npz",
         embeddings_128d=embeddings,
         embeddings_50d=emb_50d,
         embeddings_2d=emb_2d,
         cluster_labels=cluster_labels)

import csv
with open(ARTIFACTS_DIR / "cluster_assignments.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["index", "cluster", "style", "artist"])
    for i, (c, s, a) in enumerate(zip(cluster_labels, all_styles, all_artists)):
        w.writerow([i, int(c), s, a])

with open(ARTIFACTS_DIR / "clustering_metrics.txt", "w", encoding="utf-8") as f:
    f.write("=== STYLE ===\n")
    f.write(f"ARI: {ari_style:.6f}\n")
    f.write(f"NMI: {nmi_style:.6f}\n")
    f.write(f"Purity: {pur_style:.6f}\n\n")
    f.write("=== ARTIST ===\n")
    f.write(f"ARI: {ari_artist:.6f}\n")
    f.write(f"NMI: {nmi_artist:.6f}\n")
    f.write(f"Purity: {pur_artist:.6f}\n")

print("Artifacts saved in:", ARTIFACTS_DIR.resolve())

## Summary

Completed:
- Built CNN autoencoder (128D latent)
- Trained reconstruction on subset
- Extracted latent embeddings
- Reduced dimensionality (PCA 50D + 2D)
- Clustered embeddings (KMeans)
- Evaluated vs style and artist labels (ARI, NMI, Purity)

Next Steps:
- Implement irregular damage generator
- Inpainting models conditioned on clusters
- Super-resolution refinement